# Lemmatización

En este notebook incorporamos lematización al preprocesamiento del texto antes de vectorizarlo. La lematización reduce cada palabra a su forma base o lema (por ejemplo, *corriendo* → *correr*), lo que permite agrupar variantes morfológicas de una misma palabra bajo una sola representación. Usamos el modelo pequeño de spaCy para español (`es_core_news_sm`).

> **Nota:** antes de ejecutar este notebook es necesario tener instalado spaCy y el modelo de español:
> ```
> pip install spacy
> python -m spacy download es_core_news_sm
> ```

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import spacy

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 2. Carga de los datos

In [ ]:
df = pd.read_csv('train.csv')
df_eval = pd.read_csv('eval.csv')

In [ ]:
df.head()

In [ ]:
df.shape

## 3. Exploración del conjunto de datos

In [ ]:
df['decade'].value_counts().sort_index().plot(
    kind='bar', figsize=(14, 4), title='Distribución de décadas'
)
plt.xlabel('Década')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
def reporte_calidad(df):
    reporte_de_cualidad = {
        'Total records': len(df),
        'duplicated record': df.duplicated().sum(),
        'missing values': df.isnull().sum().to_dict(),
        'data types': df.dtypes.astype(str).to_dict(),
    }
    return reporte_de_cualidad

print(reporte_calidad(df))

## 4. Preprocesamiento del texto

La lematización convierte cada token a su forma canónica según el diccionario morfológico del idioma. Por ejemplo, *tenía* se reduce a *tener* y *ciudades* a *ciudad*. Esto disminuye la dimensionalidad del vocabulario y agrupa variantes de una misma palabra. En textos históricos el beneficio puede ser parcial dado que el español arcaico difiere del español moderno que el modelo de spaCy conoce.

In [ ]:
nlp = spacy.load('es_core_news_sm')

In [ ]:
def lematizar(texto):
    doc = nlp(str(texto).lower())
    return ' '.join([token.lemma_ for token in doc if not token.is_space])

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df['texto_lem'] = df['text'].apply(lematizar)
df[['text', 'texto_lem', 'decade']].head()

## 5. Partición de los datos

In [ ]:
X = df['texto_lem']
y = df['decade']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

Usamos `stratify=y` para mantener la distribución de clases.

In [ ]:
X_train.shape, X_val.shape

## 6. Construcción del pipeline

El preprocesamiento de lematización se aplicó antes de la partición y quedó almacenado en la columna `texto_lem`. El pipeline encadena directamente TF-IDF con el clasificador.

In [ ]:
pipeline_lem = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000, solver='saga')),
])

## 7. Entrenamiento con búsqueda de hiperparámetros

Exploramos el tamaño del vocabulario, la frecuencia mínima de documento, el rango de n-grams y el parámetro de regularización.

In [ ]:
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [20000, 50000],
    'tfidf__min_df': [1, 2],
    'clf__C': [0.1, 1, 10],
}

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
grid_lem = GridSearchCV(
    pipeline_lem, param_grid, cv=kfold, scoring='accuracy', n_jobs=-1
)

In [ ]:
grid_lem.fit(X_train, y_train)

In [ ]:
print('Mejores hiperparámetros:', grid_lem.best_params_)
print('Mejor score CV (accuracy):', round(grid_lem.best_score_, 4))

## 8. Evaluación del mejor modelo

In [ ]:
best_model = grid_lem.best_estimator_

y_pred_train = best_model.predict(X_train)
y_pred_val   = best_model.predict(X_val)

#### Comparación de rendimientos sobre entrenamiento y validación

In [ ]:
print('Accuracy en entrenamiento:', round(accuracy_score(y_train, y_pred_train), 4))
print('Accuracy en validación:   ', round(accuracy_score(y_val,   y_pred_val),   4))
print('Mejor score CV (accuracy):', round(grid_lem.best_score_,                  4))

In [ ]:
print(classification_report(y_val, y_pred_val))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred_val, ax=ax, colorbar=False)
plt.title('Matriz de confusión — validación')
plt.tight_layout()
plt.show()

## 9. Predicciones sobre eval.csv

In [ ]:
df_eval['texto_lem'] = df_eval['text'].apply(lematizar)

y_eval_pred = best_model.predict(df_eval['texto_lem'])

submission = pd.DataFrame({'id': df_eval['id'], 'answer': y_eval_pred})
submission.to_csv('submission_lemmatization.csv', index=False)
submission.head()